# Experiment 3: fitness-dependent movement

In [1]:
import os
from pathlib import Path

from AUTOclui import AUTOCommands as ac
from AUTOclui import runAUTO as ra
import matplotlib.pyplot as plt

from plot_3x3 import plot_equilibrium_diagram, save_3x3_plot
from pyvirtualdisplay import Display

In [2]:
folder = Path.cwd()
os.chdir(folder)

model_name = 'common_model'
output_folder = folder / 'output'
output_folder.mkdir(exist_ok=True)

parameter_file = folder / 'experiment_parameters.dat'

# Values used in the simple nine-panel loop
movement_rates = [0.1, 0.2, 0.3, 0.4, 0.5, 1, 3, 6, 10]
driver_index = 17
driver_limits = [0.0, 20.0]
start_deltar = 0.2
start_deltas = 0.02

In [3]:
display = Display(visible=False, size=(1200, 900))
display.start()

In [4]:
# Repeat the original single-plot continuation for each movement rate
runner = ra.runAUTO()
movement_panels = []

for movement_rate in movement_rates:
    print(f'\nMovement rate is {movement_rate:g}')

    # Tell common_model.f90 which habitat contrasts and movement rate to use
    parameter_file.write_text(
        f'{start_deltar} {start_deltas} {movement_rate}\n'
    )

    # First reproduce the ordinary one-parameter equilibrium continuation
    eq_forward = ac.run(
        e=model_name, c=model_name, runner=runner, NMX=4000, NPR=200
    )
    eq_backward = ac.run(
        eq_forward('EP1'), DS=-1.0e-2, runner=runner,
        NMX=4000, NPR=200, UZSTOP={14: -0.25},
    )
    equilibrium = (eq_forward + eq_backward).relabel()

    # Continue only BP1 (recovery). Follow it both ways because beta starts at 0.25.
    bp_forward = ac.run(
        equilibrium('BP1'),
        ICP=[14, driver_index], ISW=2,
        DS=1.0e-2, DSMIN=1.0e-6, DSMAX=5.0e-2,
        NMX=12000, NPR=400,
        UZSTOP={14: [-0.25, 0.5], driver_index: driver_limits},
        runner=runner,
    )
    bp_backward = ac.run(
        equilibrium('BP1'),
        ICP=[14, driver_index], ISW=2,
        DS=-1.0e-2, DSMIN=1.0e-6, DSMAX=5.0e-2,
        NMX=12000, NPR=400,
        UZSTOP={14: [-0.25, 0.5], driver_index: driver_limits},
        runner=runner,
    )
    bp_curve = (bp_forward + bp_backward).relabel()

    # Do the same for LP1: the collapse-threshold curve
    lp_forward = ac.run(
        equilibrium('LP1'),
        ICP=[14, driver_index], ISW=2,
        DS=1.0e-2, DSMIN=1.0e-6, DSMAX=5.0e-2,
        NMX=12000, NPR=400,
        UZSTOP={14: [-0.25, 0.5], driver_index: driver_limits},
        runner=runner,
    )
    lp_backward = ac.run(
        equilibrium('LP1'),
        ICP=[14, driver_index], ISW=2,
        DS=-1.0e-2, DSMIN=1.0e-6, DSMAX=5.0e-2,
        NMX=12000, NPR=400,
        UZSTOP={14: [-0.25, 0.5], driver_index: driver_limits},
        runner=runner,
    )
    lp_curve = (lp_forward + lp_backward).relabel()

    movement_panels.append({
        'movement_rate': movement_rate,
        'bp_curve': bp_curve,
        'lp_curve': lp_curve,
    })

# One additional equilibrium run supplies the Pelagic Predator 1D figure
parameter_file.write_text(f'{start_deltar} {start_deltas} 2.0\n')
reference_forward = ac.run(
    e=model_name, c=model_name, runner=runner, NMX=4000, NPR=200
)
reference_backward = ac.run(
    reference_forward('EP1'), DS=-1.0e-2, runner=runner,
    NMX=4000, NPR=200, UZSTOP={14: -0.25},
)
reference_equilibrium = (reference_forward + reference_backward).relabel()



Movement rate is 0.1
gfortran -g -fopenmp -O -c common_model.f90 -o common_model.o
gfortran -g -fopenmp -O common_model.o -o common_model.exe /auto/lib/*.o
Starting common_model ...

  BR    PT  TY  LAB       mu         L2-NORM          PL            FL            JL            PP            FP            JP      
   1     1  EP    1   0.00000E+00   9.60729E+00   0.00000E+00   5.64097E+00   0.00000E+00   0.00000E+00   7.77685E+00   0.00000E+00
   1    10  BP    2   1.20535E-01   9.60729E+00  -1.63073E-23   5.64097E+00  -1.28847E-23  -1.09256E-23   7.77685E+00  -7.13465E-24
   1    18  UZ    3   5.00000E-01   9.60729E+00  -3.45674E-33   5.64097E+00   2.63858E-32  -2.45583E-32   7.77685E+00  -9.68409E-33

  BR    PT  TY  LAB       mu         L2-NORM          PL            FL            JL            PP            FP            JP      
   2    40  LP    4   1.35290E-01   8.24557E+00   3.16132E-01   4.67911E+00   2.86026E-01   2.17414E-01   6.77063E+00   1.57906E-01
   2   200        5  

In [5]:
fig = plot_equilibrium_diagram(reference_equilibrium)
fig.savefig(
    output_folder / '1d_experiment_3_fitness_dependent_movement.png',
    dpi=200,
    bbox_inches='tight',
)
# fig.savefig(output_folder / '1d_experiment_3_fitness_dependent_movement.svg', bbox_inches='tight')
plt.close(fig)

In [6]:
# Create and save the 3x3 figure
save_3x3_plot(
    movement_panels,
    experiment=3,
    output_folder=output_folder,
    # save_svg=True,  # Uncomment to also save an SVG file
)

Saved /auto/workspace/fitness-movement/output/2d_experiment_3_fitness_dependent_movement.png


In [7]:
# Remove AUTO working files and stop the hidden display
runner.config(clean=True)
ac.clean()
display.stop()
parameter_file.unlink(missing_ok=True)

Deleting fort.* *.o *.exe *.*~ ... done
